In [1]:
from pyspark.sql import SparkSession, functions as F
s = (SparkSession.builder.master('local[2]').appName('hw22-features')
     .config('spark.sql.shuffle.partitions', '16').config('spark.ui.showConsoleProgress', 'false').getOrCreate())
s.sparkContext.setLogLevel('ERROR')
v = s.read.schema('element_uid long,user_uid long,watched_time long').option('header', True).csv('data/task3/cut_transactions.csv')
r = s.read.schema('user_uid long,element_uid long,rating int,ts double').option('header', True).csv('data/task3/cut_ratings.csv')
u = s.read.schema('user_uid long').option('header', True).csv('data/task3/ids.csv')
for name, df, keys in [('views', v, ['user_uid','element_uid']), ('ratings', r, ['user_uid','element_uid']), ('users', u, ['user_uid'])]:
    print(name, df.count(), 'повторы ключа:', df.groupBy(*keys).count().filter('count > 1').count())
    assert not df.filter(' OR '.join(k + ' IS NULL' for k in df.columns)).count()
users = u.distinct()
ratings = r.select('user_uid', 'element_uid', 'rating').distinct()
assert not ratings.groupBy('user_uid', 'element_uid').count().filter('count > 1').count()
views = v.groupBy('user_uid', 'element_uid').agg(F.sum('watched_time').alias('watched_time'))
joined = views.join(ratings.filter(F.col('rating').isin(8,9,10)), ['user_uid','element_uid']).join(users, 'user_uid', 'left_semi')
exprs = [getattr(F, op)(F.when(F.col('rating') == rating, F.col('watched_time'))).alias(f'feat_{op}_watched_time_rating_{rating}') for rating in (8,9,10) for op in ('min','max')]
result = users.join(joined.groupBy('user_uid').agg(*exprs), 'user_uid', 'left')
assert result.count() == result.select('user_uid').distinct().count() == users.count()
assert not users.exceptAll(result.select('user_uid')).count()
for rating in (8,9,10):
    lo, hi = [F.col(f'feat_{op}_watched_time_rating_{rating}') for op in ('min','max')]
    assert not result.filter((lo > hi) | (lo.isNull() != hi.isNull())).count()
result.orderBy('user_uid').toPandas().astype('Int64').to_csv('features.csv', index=False)
result.orderBy('user_uid').show(100, truncate=False)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/13 19:58:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


views 274385 повторы ключа: 0


ratings 438790 повторы ключа: 0


users 100 повторы ключа: 0


+--------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+-------------------------------+
|user_uid|feat_min_watched_time_rating_8|feat_max_watched_time_rating_8|feat_min_watched_time_rating_9|feat_max_watched_time_rating_9|feat_min_watched_time_rating_10|feat_max_watched_time_rating_10|
+--------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+-------------------------------+
|8504    |NULL                          |NULL                          |NULL                          |NULL                          |NULL                           |NULL                           |
|24124   |16                            |16                            |1575                          |2578                          |NULL                           |NULL                           |
|2525

In [2]:
checks = result.withColumn('rating_count', sum(F.col(f'feat_min_watched_time_rating_{r}').isNotNull().cast('int') for r in (8,9,10)))
checks.groupBy('rating_count').count().orderBy('rating_count').show()
for condition in ['rating_count = 0', 'rating_count = 1', 'rating_count > 1']:
    checks.filter(condition).orderBy('user_uid').show(2, truncate=False)
print('Просмотры суммируются по пользователю и фильму в исходных единицах watched_time. Точные повторы оценок удаляются, конфликтующие оценки проверяются. Отбор оценок 8–10 и выборки пользователей выполняется до агрегации признаков; left join сохраняет пользователей без просмотров и NULL.')
s.stop()


+------------+-----+
|rating_count|count|
+------------+-----+
|           0|   41|
|           1|   35|
|           2|   19|
|           3|    5|
+------------+-----+



+--------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+-------------------------------+------------+
|user_uid|feat_min_watched_time_rating_8|feat_max_watched_time_rating_8|feat_min_watched_time_rating_9|feat_max_watched_time_rating_9|feat_min_watched_time_rating_10|feat_max_watched_time_rating_10|rating_count|
+--------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+-------------------------------+------------+
|8504    |NULL                          |NULL                          |NULL                          |NULL                          |NULL                           |NULL                           |0           |
|25250   |NULL                          |NULL                          |NULL                          |NULL                          |NULL              

+--------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+-------------------------------+------------+
|user_uid|feat_min_watched_time_rating_8|feat_max_watched_time_rating_8|feat_min_watched_time_rating_9|feat_max_watched_time_rating_9|feat_min_watched_time_rating_10|feat_max_watched_time_rating_10|rating_count|
+--------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+-------------------------------+------------+
|55511   |NULL                          |NULL                          |1293                          |1293                          |NULL                           |NULL                           |1           |
|75845   |3923                          |7560                          |NULL                          |NULL                          |NULL              

+--------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+-------------------------------+------------+
|user_uid|feat_min_watched_time_rating_8|feat_max_watched_time_rating_8|feat_min_watched_time_rating_9|feat_max_watched_time_rating_9|feat_min_watched_time_rating_10|feat_max_watched_time_rating_10|rating_count|
+--------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+-------------------------------+------------+
|24124   |16                            |16                            |1575                          |2578                          |NULL                           |NULL                           |2           |
|28855   |NULL                          |NULL                          |3887                          |3887                          |7500              